# 02 — AimCLR Self-Supervised Pretraining

Pretrain a CTR-GCN backbone using SimCLR-style contrastive learning on **unlabeled** 45-frame skeleton sequences.
Produces `models/aimclr_pretrain.pth` which Phase 5 will use to initialize supervised CTR-GCN training.

**Expected outcome:** InfoNCE loss decreases from ~log(128)≈4.85 to ~1-2 over 100 epochs.

**Fallback:** If total windows < 3,000 or loss does not converge, skip this notebook and proceed directly to `03_ctrgcn_finetune.ipynb` with random initialization.

## Cell 1 — Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os

REPO_URL = 'https://github.com/michaelljx/tennis-vision-training.git'
REPO_DIR = '/content/repo'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    # Already cloned in this session — pull latest
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest changes')
else:
    # Fresh Colab session — clone
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')

sys.path.insert(0, REPO_DIR)
print(f'Repo at: {REPO_DIR}')

## Cell 2 — Install Dependencies & Check GPU

In [ ]:
import torch
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch:  {torch.__version__}')
print(f'Device:   {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected. Training will be very slow. Enable GPU in Runtime > Change runtime type.')

## Cell 3 — Load Unlabeled Data & Sanity Check

In [ ]:
# ── Configure Drive paths ──────────────────────────────────────────────────
# Adjust if your Drive folder structure differs
DRIVE_DATA   = '/content/drive/MyDrive/Tennis-CTR-GCN/data'
DRIVE_MODELS = '/content/drive/MyDrive/Tennis-CTR-GCN/models'

import os
os.makedirs(DRIVE_MODELS, exist_ok=True)

# Load unlabeled windows
unlabeled_path = f'{DRIVE_DATA}/unlabeled.npz'
data = np.load(unlabeled_path)
X_unlabeled = data['X']  # (M, T=45, V=17, C=3)

print(f'Unlabeled windows shape: {X_unlabeled.shape}')
assert X_unlabeled.shape[1:] == (45, 17, 3), f'Unexpected shape: {X_unlabeled.shape}'

M = X_unlabeled.shape[0]
print(f'Total windows:  {M}')
print(f'x range:  [{X_unlabeled[...,0].min():.3f}, {X_unlabeled[...,0].max():.3f}]')
print(f'y range:  [{X_unlabeled[...,1].min():.3f}, {X_unlabeled[...,1].max():.3f}]')
print(f'conf range: [{X_unlabeled[...,2].min():.3f}, {X_unlabeled[...,2].max():.3f}]')

if M < 3000:
    print()
    print('⚠ WARNING: fewer than 3,000 unlabeled windows.')
    print('   AimCLR benefit is limited. Consider skipping to 03_ctrgcn_finetune.ipynb.')
else:
    print(f'\n✓ Sufficient data for AimCLR pretraining ({M} windows)')

## Cell 4 — Create Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader
from aimclr.dataset import UnlabeledSkeletonDataset
from aimclr.augmentations import compose_augmentation

BATCH_SIZE = 64

dataset = UnlabeledSkeletonDataset(
    npz_path=unlabeled_path,
    augment_fn=compose_augmentation,
)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    drop_last=True,   # InfoNCE requires consistent batch size
    pin_memory=(DEVICE == 'cuda'),
)

print(f'Dataset size: {len(dataset)}')
print(f'Batches/epoch: {len(loader)}')

# Verify a single batch
v1, v2 = next(iter(loader))
print(f'View 1 shape: {v1.shape}')  # (64, 3, 45, 17)
print(f'View 2 shape: {v2.shape}')  # (64, 3, 45, 17)
assert v1.shape == (BATCH_SIZE, 3, 45, 17), f'Unexpected view shape: {v1.shape}'

## Cell 5 — Initialize CTR-GCN Backbone & AimCLR Trainer

In [ ]:
from ctrgcn.model import CTRGCN
from aimclr.trainer import AimCLRTrainer

backbone = CTRGCN(
    num_class=5,        # used in Phase 5; doesn't affect pretraining
    base_channels=64,
    adaptive=True,
)

n_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f'Backbone params: {n_params:,}')

# Quick shape test
backbone.eval()
with torch.no_grad():
    test_x = torch.randn(2, 3, 45, 17, 1)
    assert backbone(test_x).shape == (2, 5)
    assert backbone.extract_features(test_x).shape == (2, 256)
print('Shape check passed ✓')

trainer = AimCLRTrainer(
    backbone=backbone,
    feature_dim=256,
    proj_dim=128,
    temperature=0.1,
    lr=1e-3,
    weight_decay=1e-4,
    device=DEVICE,
)
print('Trainer ready.')

## Cell 6 — Train (100 Epochs)

In [ ]:
import math

EPOCHS         = 100
CKPT_PATH      = f'{DRIVE_MODELS}/aimclr_pretrain.pth'
BASELINE_LOSS  = math.log(2 * BATCH_SIZE)

print(f'Random baseline loss: log(2×{BATCH_SIZE}) = {BASELINE_LOSS:.3f}')
print(f'Target after training: < {BASELINE_LOSS * 0.5:.3f} (< 50% of baseline)')
print()

losses = trainer.train(
    loader=loader,
    epochs=EPOCHS,
    save_path=CKPT_PATH,
    print_every=10,
)

## Cell 7 — Plot Loss Curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(losses) + 1), losses, linewidth=2, label='InfoNCE Loss')
ax.axhline(y=BASELINE_LOSS, color='r', linestyle='--', alpha=0.5, label=f'Random baseline ({BASELINE_LOSS:.2f})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Avg InfoNCE Loss')
ax.set_title('AimCLR Pretraining — CTR-GCN on Tennis Skeleton Data')
ax.legend()
ax.grid(alpha=0.3)

plot_path = f'{DRIVE_MODELS}/aimclr_loss.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

final_loss = losses[-1]
print(f'\nFinal loss: {final_loss:.4f}  (baseline: {BASELINE_LOSS:.4f})')
if final_loss < BASELINE_LOSS * 0.6:
    print('✓ Loss decreased substantially — pretraining looks successful.')
else:
    print('⚠ Loss did not decrease much. Pretraining may not help Phase 5.')
    print('  Proceed to Phase 5 with random init if needed.')

## Cell 8 — (Optional) t-SNE Visualization on Val Set

In [ ]:
# Visualize whether AimCLR features cluster by stroke type on the labeled val set
from sklearn.manifold import TSNE

val_data  = np.load(f'{DRIVE_DATA}/val.npz', allow_pickle=True)
X_val     = val_data['X']   # (91, 45, 17, 3)
y_val     = val_data['y']   # (91,)
CLASS_NAMES = ['forehand', 'backhand', 'serve', 'volley', 'invalid']

backbone.eval()
with torch.no_grad():
    x_t = torch.from_numpy(X_val).permute(0, 3, 1, 2).unsqueeze(-1).float().to(DEVICE)
    feats = backbone.extract_features(x_t).cpu().numpy()  # (91, 256)

tsne = TSNE(n_components=2, perplexity=15, random_state=42, max_iter=1000)
emb  = tsne.fit_transform(feats)  # (91, 2)

colors = ['#28a745', '#17a2b8', '#ffc107', '#007bff', '#dc3545']
fig, ax = plt.subplots(figsize=(8, 6))
for i, (name, c) in enumerate(zip(CLASS_NAMES, colors)):
    mask = y_val == i
    ax.scatter(emb[mask, 0], emb[mask, 1], label=f'{name} ({mask.sum()})',
               c=c, alpha=0.8, s=40, edgecolors='white', linewidths=0.3)
ax.legend()
ax.set_title('t-SNE of AimCLR Features (Val Set — no labels used in training)')
ax.axis('off')

tsne_path = f'{DRIVE_MODELS}/aimclr_tsne.png'
plt.savefig(tsne_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {tsne_path}')
print('If clusters are visible by color → pretraining learned useful representations.')

## Cell 9 — Validate Checkpoint

In [ ]:
import os

# Verify checkpoint can be loaded
state = torch.load(CKPT_PATH, map_location='cpu')
fresh_model = CTRGCN(num_class=5)
fresh_model.load_state_dict(state)
fresh_model.eval()

with torch.no_grad():
    test_out = fresh_model(torch.randn(2, 3, 45, 17, 1))
    assert test_out.shape == (2, 5)

size_mb = os.path.getsize(CKPT_PATH) / 1e6
print(f'Checkpoint: {CKPT_PATH}')
print(f'Size:       {size_mb:.1f} MB')
print(f'Keys:       {len(state)} parameter tensors')
print()
print('✓ Checkpoint valid. Proceed to 03_ctrgcn_finetune.ipynb for supervised training.')
print()
print('Next: Upload nothing — checkpoint is already on Drive.')
print(f'  Path for Phase 5: {CKPT_PATH}')